In [1]:
import pickle

with open("X_train_tfidf_ngram.pkl", "rb") as f:
    X_train = pickle.load(f)

with open("X_test_tfidf_ngram.pkl", "rb") as f:
    X_test = pickle.load(f)

with open("y_train_tfidf_ngram.pkl", "rb") as f:
    y_train = pickle.load(f)

with open("y_test_tfidf_ngram.pkl", "rb") as f:
    y_test = pickle.load(f)

with open("tfidf_ngram.pkl", "rb") as f:
    tfidf_ngram = pickle.load(f)

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Define parameter grid for each model
param_grid = {
    "LR": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },
    "SVM Linear": {
        "model": SVC(kernel="linear", random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },
    "SVM RBF": {
        "model": SVC(kernel="rbf", random_state=42),
        "params": {"C": [0.5, 1, 2, 5], "gamma": ["scale", "auto", 0.1]}
    },
    "Multinomial NB": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.1, 0.3, 0.5, 1.0, 2.0]}
    },
    "RF": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5, 10]
        }
    }
}

print("MODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION")

# STEP 1: Hyperparameter tuning
print("\nTuning hyperparameters with 5-fold CV on training data...")

results = []
best_models = {}

for model_name, model_params in param_grid.items():
    print("\nTuning:", model_name)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=model_params["model"],
        param_grid=model_params["params"],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_train, y_train)

    best_models[model_name] = grid.best_estimator_

    # CV predictions for all metrics
    y_cv_pred = cross_val_predict(
        grid.best_estimator_,
        X_train,
        y_train,
        cv=skf,
        n_jobs=-1
    )

    cv_acc = accuracy_score(y_train, y_cv_pred)
    cv_prec = precision_score(y_train, y_cv_pred, average="binary")
    cv_rec = recall_score(y_train, y_cv_pred, average="binary")
    cv_f1 = f1_score(y_train, y_cv_pred, average="binary")

    results.append({
        "Model": model_name,
        "Best Params": grid.best_params_,
        "CV Accuracy": round(cv_acc, 4),
        "CV Precision": round(cv_prec, 4),
        "CV Recall": round(cv_rec, 4),
        "CV F1": round(cv_f1, 4),
        "Best Estimator": grid.best_estimator_
    })

    print("Best Params:", grid.best_params_)
    print("CV Accuracy :", round(cv_acc, 4))
    print("CV Precision:", round(cv_prec, 4))
    print("CV Recall   :", round(cv_rec, 4))
    print("CV F1 Score :", round(cv_f1, 4))

# STEP 2: Select best model based on CV Accuracy
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="CV Accuracy", ascending=False)

print("\nMODEL RANKING (based on 5-fold CV)")
print(df_results[[
    "Model",
    "CV Accuracy",
    "CV Precision",
    "CV Recall",
    "CV F1",
    "Best Params"
]])

best_model_name = df_results.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("\nSELECTED BEST MODEL:", best_model_name)
print("Best CV Accuracy:", df_results.iloc[0]["CV Accuracy"])

# STEP 3: Retrain best model
print("\nRETRAINING BEST MODEL ON FULL TRAINING DATA")

best_model.fit(X_train, y_train)
print("Model retrained on full training data")

# STEP 4: Evaluate on test set
print("\nFINAL EVALUATION ON TEST SET (ONE TIME ONLY!)")

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average="binary")
test_recall = recall_score(y_test, y_pred, average="binary")
test_f1 = f1_score(y_test, y_pred, average="binary")

print("\nFINAL TEST RESULTS:")
print("Test Accuracy :", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall   :", test_recall)
print("Test F1 Score :", test_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# STEP 5: Compare all models on test set
print("\nCOMPARISON WITH ALL MODELS ON TEST SET")

comparison_results = []

for model_name, estimator in best_models.items():
    estimator.fit(X_train, y_train)
    y_pred_model = estimator.predict(X_test)

    acc = accuracy_score(y_test, y_pred_model)
    prec = precision_score(y_test, y_pred_model, average="binary")
    rec = recall_score(y_test, y_pred_model, average="binary")
    f1 = f1_score(y_test, y_pred_model, average="binary")

    comparison_results.append({
        "Model": model_name,
        "Test Accuracy": round(acc, 4),
        "Test Precision": round(prec, 4),
        "Test Recall": round(rec, 4),
        "Test F1": round(f1, 4)
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(
    by=["Test F1", "Test Accuracy"],
    ascending=False
)

print("\nTEST SET PERFORMANCE OF ALL MODELS")
print(comparison_df)

best_test_model_name = comparison_df.iloc[0]["Model"]
best_test_accuracy_score = comparison_df.iloc[0]["Test Accuracy"]
best_test_precision = comparison_df.iloc[0]["Test Precision"]
best_test_recall = comparison_df.iloc[0]["Test Recall"]
best_test_f1_score = comparison_df.iloc[0]["Test F1"]

print("\nBEST MODEL BASED ON TEST SET PERFORMANCE")
print("Best Test Model:", best_test_model_name)
print("Test Accuracy :", best_test_accuracy_score)
print("Test Precision:", best_test_precision)
print("Test Recall   :", best_test_recall)
print("Test F1 Score :", best_test_f1_score)

MODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION

Tuning hyperparameters with 5-fold CV on training data...

Tuning: LR
Best Params: {'C': 5}
CV Accuracy : 0.9289
CV Precision: 0.9233
CV Recall   : 0.9355
CV F1 Score : 0.9294

Tuning: SVM Linear
Best Params: {'C': 2}
CV Accuracy : 0.927
CV Precision: 0.9232
CV Recall   : 0.9315
CV F1 Score : 0.9273

Tuning: SVM RBF
Best Params: {'C': 5, 'gamma': 'scale'}
CV Accuracy : 0.9353
CV Precision: 0.9283
CV Recall   : 0.9434
CV F1 Score : 0.9358

Tuning: Multinomial NB
Best Params: {'alpha': 0.1}
CV Accuracy : 0.9123
CV Precision: 0.8966
CV Recall   : 0.9322
CV F1 Score : 0.9141

Tuning: RF
Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
CV Accuracy : 0.9036
CV Precision: 0.8902
CV Recall   : 0.921
CV F1 Score : 0.9053

MODEL RANKING (based on 5-fold CV)
            Model  CV Accuracy  CV Precision  CV Recall   CV F1  \
2         SVM RBF       0.9353        0.9283     0.9434  0.9358   
0              LR       0.

In [3]:
import pickle

with open("best_tfidf_ngrams_covid.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved successfully")

Best model saved successfully
